# Special Ensembles

In this notebook, I'll train 3 different ensemble methods that take resampled forms of the dataset to train the models in the ensemble:

- Balanced Random Forest
- Easy Ensemble
- RUSBoostClassifier

In [1]:
import sklearn
sklearn.__version__

'1.6.1'

In [2]:
import imblearn
imblearn.__version__

'0.13.0'

In [3]:
import sys
sys.path.insert(1, '../')

In [4]:
import joblib
import pickle

from src.data import DATASETS_LS, load_dataset
from src.special_ensembles import estimator_dict
from src.hyperparams import hyperparam_special_dict
from src.cv import train_model
from src.evaluation import evaluate_model_on_test_set

In [5]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="The total space of parameters")
warnings.simplefilter(action='ignore', category=FutureWarning)

## Train models

In [6]:
# to store the results
scores_dict = {}

# training fails for the following dataset:
DATASETS_LS = [data for data in DATASETS_LS if data!="poker-8-9_vs_5"]

for dataset in DATASETS_LS:

    print(dataset)

    # initiate a dictionary per dataset
    scores_dict[dataset] = {}

    # load dataset
    X_train, X_test, y_train, y_test = load_dataset(dataset)

    # train models with hyperparameter tuning
    for estimator, params in zip(estimator_dict, hyperparam_special_dict):

      search = train_model(
          estimator_dict[estimator], 
          hyperparam_special_dict[params],
          X_train, 
          y_train,
      )

      # save model
      joblib.dump(search, f"../models/special-ensembles/{dataset}_{estimator}.pkl")

      # evaluate model
      scores_dict[dataset][estimator] = evaluate_model_on_test_set(search, X_test, y_test)


# save evaluation results
with open("../models/special-ensembles/results", "wb") as fp:
    pickle.dump(scores_dict, fp)

abalone_19
arrhythmia
car_eval_4
coil_2000
ecoli
isolet
letter_img
libras_move
mammography
oil
optical_digits
ozone_level
pen_digits
protein_homo
satimage
scene
sick_euthyroid
solar_flare_m0
spectrometer
thyroid_sick
us_crime
webpage
wine_quality
yeast_me2
cleveland-0_vs_4
dermatology-6
glass-0-1-4-6_vs_2
kddcup-buffer_overflow_vs_back
kr-vs-k-one_vs_fifteen
led7digit-0-2-4-5-6-7-8-9_vs_1
page-blocks-1-3_vs_4
pima
shuttle-2_vs_5


## Rerun for "poker-8-9_vs_5" dataset

In [7]:
from scipy import stats

from imblearn.ensemble import (
    BalancedRandomForestClassifier,
    EasyEnsembleClassifier,
    RUSBoostClassifier,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

In [8]:
dataset = "poker-8-9_vs_5"

scores_dict[dataset] = {}

X_train, X_test, y_train, y_test = load_dataset(dataset)

In [9]:
estimator = "rusboost"

rus = RUSBoostClassifier(
    estimator=DecisionTreeClassifier(random_state=10),
    n_estimators=20,
    learning_rate=1.0,
    random_state=2909,
    sampling_strategy=0.5,
)


rusboost_params = {
    "learning_rate": stats.uniform(0, 10)
}


search = train_model(
  rus, 
  rusboost_params,
  X_train, 
  y_train,
)

# save model
joblib.dump(search, f"../models/special-ensembles/{dataset}_{estimator}.pkl")

# evaluate model
scores_dict[dataset][estimator] = evaluate_model_on_test_set(search, X_test, y_test)

In [10]:
estimator = "easyEnsemble"

easy = EasyEnsembleClassifier(
    estimator = AdaBoostClassifier(random_state=10, n_estimators=10),
    n_estimators=20,
    random_state=2909,
    sampling_strategy=0.5,

)

easy_params = {
        "replacement": (True, False),
}

search = train_model(
  easy, 
  easy_params,
  X_train, 
  y_train,
)

# save model
joblib.dump(search, f"../models/special-ensembles/{dataset}_{estimator}.pkl")

# evaluate model
scores_dict[dataset][estimator] = evaluate_model_on_test_set(search, X_test, y_test)

In [11]:
estimator = "balancedRF"

search = train_model(
  estimator_dict["balancedRF"], 
  hyperparam_special_dict["brf_params"],
  X_train, 
  y_train,
)

# save model
joblib.dump(search, f"../models/special-ensembles/{dataset}_{estimator}.pkl")

# evaluate model
scores_dict[dataset][estimator] = evaluate_model_on_test_set(search, X_test, y_test)

In [12]:
# save results
with open("../models/special-ensembles/results", "wb") as fp:
    pickle.dump(scores_dict, fp)